# Financial Market Prediction — Exploratory Analysis
This notebook walks through each stage of the pipeline interactively.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import download_data
from src.features import create_features
from src.model import train_model
from src.signals import generate_signals
from src.backtest import backtest, plot_results

sns.set_theme(style='darkgrid')
%matplotlib inline

## 1. Download Data

In [ ]:
df_raw = download_data(ticker='SPY', start='2015-01-01')
df_raw.tail()

In [ ]:
df_raw['Close'].plot(figsize=(12, 4), title='SPY Close Price')
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
df = create_features(df_raw.copy())
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
df['return'].plot(ax=axes[0,0], title='Daily Returns')
df[['ma10','ma30']].plot(ax=axes[0,1], title='Moving Averages')
df['volatility'].plot(ax=axes[1,0], title='Rolling Volatility')
df['target'].plot(ax=axes[1,1], title='Target (5-day forward return)')
plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
corr_cols = ['return', 'ma10', 'ma30', 'ma_ratio', 'volatility', 'target']
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Train Model

In [ ]:
model, metrics = train_model(df)
print('Metrics:', metrics)

In [ ]:
# Feature importances
import pandas as pd
importances = pd.Series(model.feature_importances_, index=metrics['features'])
importances.sort_values().plot(kind='barh', figsize=(8, 4), title='Feature Importances')
plt.tight_layout()
plt.show()

## 5. Generate Signals & Backtest

In [ ]:
df = generate_signals(df, model)
strategy, market = backtest(df)
plot_results(strategy, market, title='SPY — ML Strategy vs Buy & Hold')